<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/GINfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

# FIX: define filenames correctly
edge_file = "cora.cites"
content_file = "cora.content"

print("Files loaded:", uploaded.keys())

Saving cora.cites to cora (2).cites
Saving cora.content to cora (3).content
Files loaded: dict_keys(['cora (2).cites', 'cora (3).content'])


In [ ]:
print(edge_file)
print(content_file)

cora.cites
cora.content


In [ ]:
import networkx as nx
import numpy as np
import torch

G = nx.Graph()

# edges
with open(edge_file, "r") as f:
    for line in f:
        u, v = line.strip().split()
        G.add_edge(int(u), int(v))

# features
features = []
node_ids = []

with open(content_file, "r") as f:
    for line in f:
        parts = line.strip().split()
        node_ids.append(int(parts[0]))
        features.append([int(x) for x in parts[1:-1]])

features = np.array(features)

# mapping
id_map = {nid: i for i, nid in enumerate(node_ids)}

# keep only nodes present in graph
node_list = list(G.nodes())
node_index = {n: id_map[n] for n in node_list}

x = torch.tensor(features, dtype=torch.float)

# normalize (IMPORTANT)
x = (x - x.mean(0)) / (x.std(0) + 1e-6)

print("Nodes:", G.number_of_nodes())

Nodes: 2708


In [ ]:
import random

edges = list(G.edges())
random.seed(42)
random.shuffle(edges)

num_test = int(0.1 * len(edges))

test_pos = edges[:num_test]
train_edges = edges[num_test:]

In [ ]:
def sample_neg_edges(G, num_samples):
    neg = set()
    nodes = list(G.nodes())
    while len(neg) < num_samples:
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u != v and not G.has_edge(u, v):
            neg.add((u, v))
    return list(neg)

train_neg = sample_neg_edges(G, len(train_edges))
test_neg = sample_neg_edges(G, len(test_pos))

test_edges = [(u, v, 1) for (u, v) in test_pos] + \
             [(u, v, 0) for (u, v) in test_neg]

In [ ]:
from torch_geometric.data import Data

edge_index = torch.tensor(
    [[id_map[u], id_map[v]] for u, v in train_edges] +
    [[id_map[v], id_map[u]] for u, v in train_edges],
    dtype=torch.long
).t()

data = Data(x=x, edge_index=edge_index)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GINConv

class GIN(nn.Module):
    def __init__(self, in_dim, hidden_dim=256):
        super().__init__()

        def mlp(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Linear(out_dim, out_dim)
            )

        self.conv1 = GINConv(mlp(in_dim, hidden_dim))
        self.conv2 = GINConv(mlp(hidden_dim, hidden_dim))
        self.conv3 = GINConv(mlp(hidden_dim, hidden_dim))

        self.dropout = nn.Dropout(0.5)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        return x

In [ ]:
class LinkPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, z_u, z_v):
        return self.mlp(torch.cat([z_u, z_v], dim=-1)).squeeze()

In [ ]:
import torch
import torch.nn.functional as F

model = GIN(in_dim=x.shape[1])
predictor = LinkPredictor()

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(predictor.parameters()),
    lr=0.001,
    weight_decay=1e-5
)

def train():
    model.train()
    predictor.train()

    optimizer.zero_grad()

    z = model(data.x, data.edge_index)

    pos_loss = 0
    neg_loss = 0

    # positive
    for u, v in train_edges:
        score = predictor(z[id_map[u]], z[id_map[v]])
        pos_loss += F.binary_cross_entropy_with_logits(score, torch.tensor(1.0))

    # negative
    for u, v in train_neg:
        score = predictor(z[id_map[u]], z[id_map[v]])
        neg_loss += F.binary_cross_entropy_with_logits(score, torch.tensor(0.0))

    loss = pos_loss + neg_loss
    loss.backward()
    optimizer.step()

    return loss.item()

for epoch in range(100):
    loss = train()
    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss)

Epoch: 0 Loss: 6267.6689453125
Epoch: 10 Loss: 5819.11962890625
Epoch: 20 Loss: 5853.30908203125
Epoch: 30 Loss: 5887.5986328125
Epoch: 40 Loss: 5089.751953125
Epoch: 50 Loss: 5497.76416015625
Epoch: 60 Loss: 4823.796875
Epoch: 70 Loss: 4143.75439453125
Epoch: 80 Loss: 4214.76220703125
Epoch: 90 Loss: 3952.68603515625


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

model.eval()
predictor.eval()

z = model(data.x, data.edge_index)

y_true = []
y_score = []

for u, v, label in test_edges:
    score = predictor(z[id_map[u]], z[id_map[v]])
    y_true.append(label)
    y_score.append(torch.sigmoid(score).item())

ap = average_precision_score(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

print("GIN Improved Results:")
print("AP :", round(ap, 4))
print("AUC:", round(auc, 4))

GIN Improved Results:
AP : 0.751
AUC: 0.7442
